Below is a claims pipeline that relies upon longformer-base-4096
- handles full claim narratives (up to 8k tokens fits in 4Gb of memory and is fast & high accuracy).
- [ vs Clinical‑Longformer which is slower. Bio_ClinicalBERT 256 tokens & fast & lower accuracy.]

## 00 - imports and device

In [1]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    LongformerForSequenceClassification
)

import shap
import matplotlib.pyplot as plt
import seaborn as sns

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [2]:
# windows or wsl
import platform
print(platform.system())
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Windows
2.12.0.dev20260408+cu128
12.8
True
NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
%matplotlib inline

## 01 - Load your dataset

In [4]:
df = pd.read_json("synthetic_claims.json")
df.head()

,id,severity,theme,department,claim_text
0,1,high,diagnostic_error,ED/Cardiology,"The patient, a 68-year-old male with hypertens..."
1,2,high,failure_to_escalate,Surgery/ICU,A 54-year-old female underwent elective laparo...
2,3,low,diagnostic_error,ED/Fracture Clinic,A 32-year-old male attended A&E after falling ...
3,4,high,delay_in_treatment,ED/Respiratory,A 76-year-old patient with COPD presented with...
4,5,high,fetal_monitoring_failure,Maternity,A 29-year-old primigravida presented in labour...


## 02 - Map severity

In [5]:
severity_map = {"low": 0, "moderate": 1, "high": 2}
df["severity_label"] = df["severity"].map(severity_map)

## 03 - train/validation split

In [6]:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [7]:
train_df, val_df

(    id  severity                  theme   department  \
 55  56  moderate  communication_failure           GP   
 88  89      high         surgical_error      Theatre   
 26  27      high       diagnostic_error           ED   
 42  43      high    failure_to_escalate           ED   
 69  70       low   administrative_delay  Outpatients   
 ..  ..       ...                    ...          ...   
 60  61      high       diagnostic_error           ED   
 71  72  moderate       medication_error     Pharmacy   
 14  15      high       diagnostic_error           ED   
 92  93      high    failure_to_escalate           ED   
 51  52  moderate       medication_error     Pharmacy   
 
                                            claim_text  severity_label  
 55  A patient was not informed of abnormal liver f...               1  
 88  A surgical instrument was retained during proc...               2  
 26  A 39-year-old female presented with chest pain...               2  
 42  A patient with se

## 04 - Convert to huggingface dataset

In [8]:
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

## 05 Load tokenisers and models

In [9]:
# create a wrapper for the model
class LongformerForClaims(LongformerForSequenceClassification):
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # Inject global attention mask
        global_attention_mask = torch.zeros_like(input_ids)
        global_attention_mask[:, 0] = 1 # CLS token global

        return super().forward(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask,
            labels=labels,
            output_attentions=kwargs.get("output_attentions", False)
        )

In [10]:
LONGFORMER_NAME = "allenai/longformer-base-4096"

long_tokenizer = AutoTokenizer.from_pretrained(LONGFORMER_NAME)
# load wrapped model
long_model = LongformerForClaims.from_pretrained(LONGFORMER_NAME, num_labels=3).to(DEVICE)


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\marke\.cache\huggingface\hub\models--allenai--longformer-base-4096. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Some weights of LongformerForClaims were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 06 - Tokenisation functions

In [11]:
def tokenize_longformer(batch):
    return long_tokenizer(
        batch["claim_text"],
        padding="max_length",
        truncation=True,
        max_length=2048
    )

## 07 - Apply tokenisation

In [12]:
long_train = train_ds.map(tokenize_longformer, batched=True)
long_val = val_ds.map(tokenize_longformer, batched=True)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

## 08 - Rename label column

In [13]:
long_train = long_train.rename_column("severity_label", "labels")
long_val = long_val.rename_column("severity_label", "labels")

## 09 - Remove unused columns

In [14]:
cols_to_remove = ["id", "severity", "theme", "department", "claim_text"]

long_train = long_train.remove_columns(cols_to_remove)
long_val = long_val.remove_columns(cols_to_remove)

## 10 - Set format for pytorch

In [15]:
long_train.set_format("torch")
long_val.set_format("torch")

## 11 - Metrics

In [16]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

## 12 - Training arguments

In [17]:
# work on an old GPU
long_args = TrainingArguments(
    output_dir="./longformer-claims",
    num_train_epochs=20,
    per_device_train_batch_size=1,     # ← FIXED
    per_device_eval_batch_size=1,      # ← FIXED
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                # save space
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    warmup_steps=200,                   # more warm up steps
    max_grad_norm=1.0,
    gradient_checkpointing=True,        # ← HUGE FIX
    fp16=True                         # floating point precision
)

## 13 - Trainers

In [18]:
long_trainer = Trainer(
    model=long_model,
    args=long_args,
    train_dataset=long_train,
    eval_dataset=long_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

## 14 - train and evaluation

In [19]:
long_trainer.train()
long_metrics = long_trainer.evaluate()

C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.060700,1.004810,0.500000,0.222222
2,0.931900,0.609123,0.800000,0.544444
3,0.537100,0.334340,0.950000,0.964519
4,0.114100,0.431445,0.950000,0.964519


C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
C:\Users\marke\projects\gpu-transformers\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1269: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variant

In [20]:
print("=== MODEL PERFORMANCE  ===")

print(f"Longformer Accuracy:{long_metrics['eval_accuracy']:.3f}")
print(f"Longformer F1:      {long_metrics['eval_f1_macro']:.3f}")

print("\nInterpretation:")
print("- Longformer processes full claim narratives → more stable severity predictions.")

=== MODEL PERFORMANCE  ===
Longformer Accuracy:0.950
Longformer F1:      0.965

Interpretation:
- Longformer processes full claim narratives → more stable severity predictions.


In [21]:
# check which one it got wrong
long_preds = long_trainer.predict(long_val)
long_pred_labels = np.argmax(long_preds.predictions, axis=-1)
long_true_labels = long_preds.label_ids

long_misclassified_idx = np.where(long_pred_labels != long_true_labels)[0]
long_misclassified = val_df.iloc[long_misclassified_idx]
long_misclassified

,id,severity,theme,department,claim_text,severity_label
10,11,moderate,diagnostic_error,ED,A 58-year-old male presented with palpitations...,1


## 15 - shap explainer - longformer

In [22]:
# predict from masked text
def longformer_predict_proba(masked_texts):
    # SHAP passes List[List[str]] → convert back to strings
    texts = [" ".join(tokens) for tokens in masked_texts]

    enc = long_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=2048,
        return_tensors="pt"
    ).to(DEVICE)

    # global attention mask (CLS attends globally)
    global_attention_mask = torch.zeros_like(enc["input_ids"])
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        outputs = long_model(
            **enc,
            global_attention_mask=global_attention_mask
        )
        probs = torch.softmax(outputs.logits, dim=-1)

    return probs.cpu().numpy()   # shape: (batch, 3)

In [23]:
# 2. Build SHAP explainer (partition algorithm)
long_explainer = shap.Explainer(
    longformer_predict_proba,
    long_tokenizer,
    algorithm="partition"
)

In [24]:
# 4. Explain a claim
#claim = df["claim_text"].iloc[0]
# check the one it got wrong in the validation set
claim_text = val_df.iloc[0]["claim_text"] 
long_shap_values = long_explainer([claim_text], max_evals=200)

Input ids are automatically padded to be a multiple of `config.attention_window`: 512


  0%|          | 0/198 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:18, 18.86s/it]               


In [25]:
# get the predicted class
pred_class = longformer_predict_proba([claim_text]).argmax()
severity_map = {0: "low", 1: "moderate", 2: "high"}
print("Predicted severity:", severity_map[pred_class])

Predicted severity: high


In [26]:
# plot the shap for that class
shap.plots.text(long_shap_values[0, :, pred_class])

In [27]:
# plot the shap for that class
shap.plots.text(long_shap_values[0, :, 0])

In [28]:
# plot the shap for that class
shap.plots.text(long_shap_values[0, :, pred_class])

In [29]:
# 5. Visualise
shap.plots.text(long_shap_values[0])